# Data Pipeline
## Loading e-SNLI dataset


In [1]:
from IPython.display import display, IFrame
from google.colab import userdata
from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_file
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer

import numpy as np
import os
import plotly.express as px
import torch
import torch.nn as nn

# Setup
## Retrieve HF token
This will be different based on whether we run Colab vs locally.

In [18]:
import sys # Check if in Colab Env.
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Login into HF using HF token.
    hf_token = userdata.get('HFWrite')
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

## Set `device` parameter

In [ ]:
torch.set_grad_enabled(False) # avoid blowing up mem
if torch.backends.mps.is_available():
  device = 'mps'
elif torch.cuda.is_available():
  device = 'cuda'
else:
  device = 'cpu'

print(f'Device: {device}')

# Data Setup
## Import
Import e-SNLI data files. The train data is split across two files. We import both and merge the datasets.

In [3]:
import pandas as pd

# URLs to the raw files on GitHub
url1 = "https://raw.githubusercontent.com/OanaMariaCamburu/e-SNLI/master/dataset/esnli_train_1.csv"
url2 = "https://raw.githubusercontent.com/OanaMariaCamburu/e-SNLI/master/dataset/esnli_train_2.csv"

print("Downloading part 1...")
df1 = pd.read_csv(url1)

print("Downloading part 2...")
df2 = pd.read_csv(url2)

# Merge the two dataframes
print("Merging datasets...")
esnli_train = pd.concat([df1, df2], axis=0, ignore_index=True)

# Save the merged file to the Colab local directory
esnli_train.to_csv("ensli_train.csv", index=False)
print("Done! Saved as 'ensli_train.csv'.")

Merging datasets...
Done! Saved as 'ensli_train.csv'.


In [4]:
esnli_train.columns

Index(['pairID', 'gold_label', 'Sentence1', 'Sentence2', 'Explanation_1',
       'WorkerId', 'Sentence1_marked_1', 'Sentence2_marked_1',
       'Sentence1_Highlighted_1', 'Sentence2_Highlighted_1'],
      dtype='object')

In [5]:
interesting_columns = ['Sentence1', 'Sentence2', 'Sentence1_marked_1', 'Sentence2_marked_1', 'Explanation_1', 'gold_label']
print("\nTop 5 rows:")
esnli_train[interesting_columns].head()


Top 5 rows:


,Sentence1,Sentence2,Sentence1_marked_1,Sentence2_marked_1,Explanation_1,gold_label
0,A person on a horse jumps over a broken down a...,A person is training his horse for a competition.,A person on a horse jumps over a broken down a...,A person is *training* *his* *horse* for a co...,the person is not necessarily training his horse,neutral
1,A person on a horse jumps over a broken down a...,"A person is at a diner, ordering an omelette.",A person *on* *a* *horse* *jumps* over a brok...,"A person *is* *at* *a* *diner,* *ordering* an...",One cannot be on a jumping horse cannot be a d...,contradiction
2,A person on a horse jumps over a broken down a...,"A person is outdoors, on a horse.",A person on a horse jumps over *a* *broken* *...,"A person is *outdoors,* on a horse.",a broken down airplane is outdoors,entailment
3,Children smiling and waving at camera,They are smiling at their parents,Children smiling and waving at camera,They are smiling *at* *their* *parents*,Just because they are smiling and waving at a ...,neutral
4,Children smiling and waving at camera,There are children present,*Children* *smiling* *and* *waving* at camera,There are children *present*,The children must be present to see them smili...,entailment


## Generate Instruction Tuning Prompt
Generate a prompt for instruction tuning by providing examples of Neutral, Entailment and Contradiction.

In [15]:
# 1. Get all the unique labels.
unique_labels = esnli_train['gold_label'].unique()
print(f"Labels: {', '.join(unique_labels)}\n")

# 2. Get one row per label.
unique_samples = esnli_train.drop_duplicates(subset=['gold_label']).copy()

# 3. Format the contents of the filtered rows
# We use .apply with axis=1 here because it's easier to handle the "Example N" numbering later
unique_samples['formatted_input'] = unique_samples.apply(
    lambda x: f"A: {x['Sentence1']}\nB: {x['Sentence2']}\n{x['Explanation_1']}\nLabel: {x['gold_label']}",
    axis=1
)

# 4. Print the newly formatted string for the first row for verification
print("--- Verification of First Row ---")
print(repr(unique_samples['formatted_input'].iloc[0]))
print("\n" + "-"*30 + "\n")

# 5. Merge all rows into a single string with "Example N" headers
it_prompt = "\n".join(
    [f"Example {i+1}:\n{text}" for i, text in enumerate(unique_samples['formatted_input'])]
)

print(f"#Final Merged String#\n{it_prompt}")

Labels: neutral, contradiction, entailment

--- Verification of First Row ---
'A: A person on a horse jumps over a broken down airplane.\nB: A person is training his horse for a competition.\nthe person is not necessarily training his horse\nLabel: neutral'

------------------------------

#Final Merged String#
Example 1:
A: A person on a horse jumps over a broken down airplane.
B: A person is training his horse for a competition.
the person is not necessarily training his horse
Label: neutral
Example 2:
A: A person on a horse jumps over a broken down airplane.
B: A person is at a diner, ordering an omelette.
One cannot be on a jumping horse cannot be a diner ordering food.
Label: contradiction
Example 3:
A: A person on a horse jumps over a broken down airplane.
B: A person is outdoors, on a horse.
a broken down airplane is outdoors
Label: entailment


## Generate Lean Prompt
Lean prompt with instructions but no examples.

In [19]:
lean_prompt = 'Determine if statement B is an entailment, contradiction or neutral. Reason step by step and finally provide a one-word answer.\n'

In [29]:
x = tokenizer.encode(lean_prompt, add_special_tokens=True)

[2,
 102752,
 768,
 5456,
 603,
 563,
 614,
 83155,
 658,
 236764,
 38912,
 653,
 12643,
 236761,
 44191,
 2918,
 684,
 2918,
 532,
 6861,
 2847,
 496,
 886,
 236772,
 3017,
 3890,
 236761,
 107]

In [32]:
esnli_train['prompt'] = (
    '<start_of_turn>user ' +
    lean_prompt + it_prompt + '\n'
    'A: ' + esnli_train['Sentence1'] + '\n' +
    'B: ' + esnli_train['Sentence2'] + '\n' +
    '<end_of_turn>model'
)



esnli_train['prompt_token_length'] = esnli_train['prompt'].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True)) if isinstance(x, str) else 0)

print("--- Verification of First Row ---")
print(f"Length: {esnli_train['prompt_token_length'].iloc[0]}")
print(esnli_train['prompt'].iloc[0])

--- Verification of First Row ---
Length: 204
<start_of_turn>user Determine if statement B is an entailment, contradiction or neutral. Reason step by step and finally provide a one-word answer.
Example 1:
A: A person on a horse jumps over a broken down airplane.
B: A person is training his horse for a competition.
the person is not necessarily training his horse
Label: neutral
Example 2:
A: A person on a horse jumps over a broken down airplane.
B: A person is at a diner, ordering an omelette.
One cannot be on a jumping horse cannot be a diner ordering food.
Label: contradiction
Example 3:
A: A person on a horse jumps over a broken down airplane.
B: A person is outdoors, on a horse.
a broken down airplane is outdoors
Label: entailment
A: A person on a horse jumps over a broken down airplane.
B: A person is training his horse for a competition.
<end_of_turn>model


# Loading the Model
Loading Gemma 3 1B model.

In [9]:
_PRETRAINED_MODEL_NAME = 'google/gemma-3-4b-it'
_SAE_REPO_ID = 'google/gemma-scope-2-4b-it'
_RELEASE_NAME = 'gemma-scope-2-4b-it-res'
_SAE_ID = 'layer_22_width_65k_l0_medium'

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    _PRETRAINED_MODEL_NAME,
    device_map='auto',
)
tokenizer =  AutoTokenizer.from_pretrained(_PRETRAINED_MODEL_NAME)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Run model with a test prompt.

In [36]:
verification_prompt = esnli_train['prompt'].iloc[3]
verification_label = esnli_train['gold_label'].iloc[3]
enc_inputs = tokenizer.encode(verification_prompt, return_tensors='pt', add_special_tokens=True).to('cuda')
input_token_length = enc_inputs.shape[1]
if input_token_length == esnli_train['prompt_token_length'].iloc[3]:
  print(f'Verification prompt = input token length = {input_token_length}')
else:
  print(f'Verification prompt = {esnli_train['prompt_token_length'].iloc[3]} but input token length = {input_token_length}')
outputs = model.generate(input_ids=enc_inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0]))
print('---------------')
print(tokenizer.decode(outputs[0, input_token_length:]))
print('---------------')
print(f'Actual label: {verification_label}')

Verification prompt = input token length = 194
Input tokens length: 194
---------------
<bos><start_of_turn>user Determine if statement B is an entailment, contradiction or neutral. Reason step by step and finally provide a one-word answer.
Example 1:
A: A person on a horse jumps over a broken down airplane.
B: A person is training his horse for a competition.
the person is not necessarily training his horse
Label: neutral
Example 2:
A: A person on a horse jumps over a broken down airplane.
B: A person is at a diner, ordering an omelette.
One cannot be on a jumping horse cannot be a diner ordering food.
Label: contradiction
Example 3:
A: A person on a horse jumps over a broken down airplane.
B: A person is outdoors, on a horse.
a broken down airplane is outdoors
Label: entailment
A: Children smiling and waving at camera
B: They are smiling at their parents
<end_of_turn>model
The statement B follows logically from statement A. If children are smiling and waving at a camera, it is reason

In [50]:
## Measure Accuracy.
inputs = tokenizer(
    esnli_train['prompt'].astype(str).tolist(),
    padding=True,
    add_special_tokens=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=256,
        pad_token_id=tokenizer.pad_token_id
    )

decoded_outputs = tokenizer.batch_decode(
    output_tokens, skip_special_tokens=True)

decoded_outputs[:5]

OutOfMemoryError: CUDA out of memory. Tried to allocate 796.36 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.60 GiB is free. Process 6735 has 13.14 GiB memory in use. Of the allocated memory 13.00 GiB is allocated by PyTorch, and 17.16 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# SAE Activations
Hook GemmaScope 2 SAE and inspect the activations.

In [24]:
# SAE Lens doesn't provide an implementation for Gemma-3-1b yet.
class JumpReLUSAE(nn.Module):
  def __init__(self, d_in, d_sae, affine_skip_connection=False):
    # Encoder.
    super().__init__()
    self.w_enc = nn.Parameter(torch.zeros(d_in, d_sae))
    self.b_enc = nn.Parameter(torch.zeros(d_sae))
    self.threshold = nn.Parameter(torch.zeros(d_sae))

    # Decoder.
    self.w_dec = nn.Parameter(torch.zeros(d_sae, d_in))
    self.b_dec = nn.Parameter(torch.zeros(d_in))

    if affine_skip_connection:
      self.affine_skip_connection = nn.Parameter(torch.zeros(d_in, d_in))
    else:
      self.affine_skip_connection = None

  def encode(self, input_acts):
    print(f'W_enc type: {self.w_enc.dtype}')
    print(f'b_enc type: {self.b_enc.dtype}')
    pre_acts = input_acts @ self.w_enc + self.b_enc
    mask = (pre_acts > self.threshold)
    acts = mask * torch.nn.functional.relu(pre_acts)
    return acts

  def decode(self, acts):
    return acts @ self.w_dec + self.b_dec

  def forward(self, x):
    acts = self.encode(x)
    recon = self.decode(acts)
    if self.affine_skip_connection is not None:
      return recon + x @ self.affine_skip_connection
    return recon

In [15]:
path_to_params = hf_hub_download(
    repo_id=_SAE_REPO_ID,
    filename=f'resid_post/{_SAE_ID}/params.safetensors'
)

params = load_file(path_to_params)
print(f"\nSAE parameters:")
for name, tensor in params.items():
    print(f"  {name}: {tensor.shape}")

resid_post/layer_22_width_65k_l0_medium/(…):   0%|          | 0.00/1.34G [00:00<?, ?B/s]


SAE parameters:
  b_dec: torch.Size([2560])
  b_enc: torch.Size([65536])
  threshold: torch.Size([65536])
  w_dec: torch.Size([65536, 2560])
  w_enc: torch.Size([2560, 65536])


In [25]:
d_model, d_sae = params['w_enc'].shape
sae = JumpReLUSAE(d_model, d_sae)
sae.load_state_dict(params)
sae.cuda()

JumpReLUSAE()

In [21]:
def gather_activations(model, target_layer, inputs):
  """Returns the activations in the target_layer of the model using inputs.

    Args:
      model: The HF model to use.
      target_layer: int, index of target layer.
      inputs: torch.Tensor, input to the model.

    Returns:
      torch.Tensor, activations in the target layer.
  """
  cache = {}

  def hook_fn(module, input, output):
      # Hugging Face models often return tuples (hidden_states, attention_weights, etc.)
      # We usually want the first element which is the hidden states.
      if isinstance(output, tuple):
          activations = output[0]
      else: # If it's just a tensor
          activations = output

      # Detach to ensure we don't keep the computation graph
      cache['activations'] = activations.detach()

  # Register the hook
  handle = model.model.language_model.layers[target_layer].register_forward_hook(hook_fn)

  try:
      with torch.no_grad():
          model(inputs)
  finally:
      # Ensure the hook is removed
      handle.remove()

  return cache['activations']

In [27]:
# --- Quick single-prompt demo: gather activations for GENERATED tokens only ---

TARGET_LAYER = 22  # must match the SAE (layer_22_width_65k_l0_medium)
MAX_NEW_TOKENS = 256

# 1. Tokenize the prompt
sample_prompt = esnli_train['prompt'].iloc[0]
prompt_ids = tokenizer.encode(sample_prompt, return_tensors='pt', add_special_tokens=True).to(device)
prompt_len = prompt_ids.shape[1]
print(f'Prompt tokens: {prompt_len}')

# 2. Generate the model's response (returns prompt + generated tokens)
full_ids = model.generate(input_ids=prompt_ids, max_new_tokens=MAX_NEW_TOKENS)
gen_len = full_ids.shape[1] - prompt_len
print(f'Generated tokens: {gen_len}')
print(f'Full sequence tokens: {full_ids.shape[1]}')

# 3. Run a forward pass on the full sequence and hook layer 22
residual_acts = gather_activations(model, TARGET_LAYER, full_ids)
print(f'Residual activations shape (full): {residual_acts.shape}')  # (1, prompt+gen, d_model)

# 4. Slice to keep only the generated-token activations
gen_acts = residual_acts[:, prompt_len:, :]
print(f'Generated-only activations shape: {gen_acts.shape}')  # (1, gen_len, d_model)
print(f'Type: {gen_acts.dtype}')

# 5. Encode through the JumpReLU SAE to get sparse features
sae_feature_acts = sae.encode(gen_acts.to(torch.float32))
print(f'SAE feature activations shape: {sae_feature_acts.shape}')  # (1, gen_len, d_sae)

# 6. Inspect features at the last generated token
last_token_features = sae_feature_acts[0, -1, :]  # (d_sae,)
n_active = (last_token_features > 0).sum().item()
print(f'\nActive features at last generated token: {n_active} / {last_token_features.shape[0]}')

# 7. Show the top-k most activated features
TOP_K = 10
top_vals, top_idxs = last_token_features.topk(TOP_K)
print(f'\nTop {TOP_K} SAE features at last generated token:')
for rank, (idx, val) in enumerate(zip(top_idxs.tolist(), top_vals.tolist()), 1):
    print(f'  #{rank}  feature {idx:>5d}  activation = {val:.4f}')

# 8. Print the generated text for reference
print(f'\n--- Generated Response ---')
print(tokenizer.decode(full_ids[0, prompt_len:], skip_special_tokens=True))

Prompt tokens: 71
Generated tokens: 256
Full sequence tokens: 327
Residual activations shape (full): torch.Size([1, 327, 2560])
Generated-only activations shape: torch.Size([1, 256, 2560])
Type: torch.bfloat16
W_enc type: torch.float32
b_enc type: torch.float32
SAE feature activations shape: torch.Size([1, 256, 65536])

Active features at last generated token: 48 / 65536

Top 10 SAE features at last generated token:
  #1  feature   514  activation = 1825.6299
  #2  feature  2288  activation = 1518.4009
  #3  feature   977  activation = 1043.5393
  #4  feature  1778  activation = 1024.3516
  #5  feature   401  activation = 964.8574
  #6  feature  1390  activation = 941.4670
  #7  feature  2635  activation = 636.9327
  #8  feature  1810  activation = 626.5433
  #9  feature   459  activation = 582.5459
  #10  feature  2693  activation = 582.2654

--- Generated Response ---

Okay, let's break down this entailment/contradiction/neutral analysis step-by-step:

**1. Understanding the Statemen

In [ ]:
px.line(
    cache['blocks.22.hook_resid_post.hook_sae_acts_post'][0, -1, :].cpu().numpy(),
    title='Feature activations at the final token position',
    labels={'index': 'Featutre', 'value': 'Activation'},
).show()

html_template = 'https://www.neuronpedia.org/gemma-3-1b-it/22-gemmascope-2-res-65k/{}?embed=true&embedexplanation=true&embedplots=true&embedsteer=true&embedactivations=true&embedlink=true&embedtest=true'

vals, inds = torch.topk(
    cache['blocks.22.hook_resid_post.hook_sae_acts_post'][0, -1, :], 5
)

for val, ind in zip(vals, inds):
    print(f'Feature {ind} fired {val:.2f}')
    html = html_template.format(ind)
    print('URL: ' + html)
    display(IFrame(html, width=1200, height=300))